# Fase 1 · Definición del problema y entorno reproducible

**Proyecto:** Fallas recurrentes de transmisión en la red de un operador móvil (NOC de WOM, octubre de 2026)
**Curso:** MCDI500 · Programación para la Ciencia de Datos · Magíster en Ciencia de Datos e IA, UNAB
**Integrante:** Edgar Arcia Velásquez (Grupo 6, trabajo individual) · **Docente:** Dr. Omar Salinas Silva

---

### Qué produce esta fase

La Fase 2 recibe un conjunto de datos y lo deja listo para analizar. Para eso, la Fase 1 deja resuelto:

1. **El problema escrito** como estructura de datos: pregunta, objetivos, alcance, restricciones y supuestos.
2. **Un entorno verificado** con evidencia: el intérprete del proyecto y las versiones de las librerías clave.
3. **La estructura del repositorio** y los archivos que lo hacen reproducible.
4. **Un módulo propio** (`src/utilidades.py`), probado con casos normales, límite y excepciones.
5. **La ficha y el diccionario del conjunto de datos**, con el rol analítico de cada variable.

> **Alcance de este notebook:** aquí no se limpia, no se imputa y no se transforma. Eso es la Fase 2.
> Esta fase solo **mide** lo que trae el archivo, para que la Fase 2 decida con cifras.

## Introducción

El centro de operaciones de red (NOC) de un operador móvil registra cada falla de la red como una
incidencia con número de ticket (TT). En octubre de 2026 se registraron **21.542 incidencias**.
Algunas se normalizan en minutos y otras tardan días.

Las fallas de **transmisión** son pocas, pero son las que más dejan sitios fuera de servicio. Un
sitio pierde su transmisión cuando se cae su enlace de microondas (MW), se corta la fibra propia
(FOP) o falla la fibra que arrienda a un proveedor. Si un mismo sitio falla una y otra
vez, el problema no se resuelve atendiendo cada incidencia por separado. Hay que escalarlo al área
que corresponde (transmisión propia o el proveedor) y evaluar un cambio de transmisión, por ejemplo
un enlace de respaldo o un cambio de medio. Para eso hace falta saber, con datos, **qué elementos
fallan más, cuánto tiempo acumulan sin servicio y qué tipo de enlace los deja caer**.

### Herramientas del ecosistema científico

| Pieza | Para qué se usa | Qué falla si se omite |
|---|---|---|
| Entorno virtual (`.venv`) | Aislar las versiones de las librerías del proyecto | El notebook da otro resultado en otro equipo |
| `requirements.txt` | Reconstruir el mismo entorno con un comando | Nadie puede reproducir las cifras |
| pandas y NumPy | Leer, medir y transformar la tabla | No hay forma práctica de procesar 21 mil filas |
| Jupyter | Juntar el texto, el código y la evidencia | El razonamiento queda separado del cálculo |
| Git y GitHub | Historial de cambios con autor, fecha y motivo | No se puede demostrar cómo avanzó el trabajo |

In [1]:
import sys                      # intérprete en uso: se verifica en la sección 2
import json                     # exportación de metadatos
import platform                 # sistema operativo, para el registro del entorno
import subprocess               # consultas a Git desde el notebook (sección 6)
import importlib                # recarga del módulo propio
from datetime import date
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

import numpy as np
import pandas as pd

SEMILLA = 42                    # semilla fija: cualquier resultado aleatorio se puede repetir
np.random.seed(SEMILLA)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
print("Librerías cargadas. Semilla:", SEMILLA)

Librerías cargadas. Semilla: 42


## 1. Definición del problema

**Qué hace este paso.** Deja el problema escrito como un **diccionario de Python** y no como un
párrafo suelto. Lo que el informe declara (problemática, objetivos, alcance y supuestos) sale de
aquí, y así el informe y el notebook no pueden contradecirse.

In [2]:
PROYECTO = {
    "titulo": "Fallas recurrentes de transmisión en la red de un operador móvil",
    "grupo": "Grupo 6 (trabajo individual, informado al docente)",
    "asignatura": "MCDI500 · Programación para la Ciencia de Datos",
    "integrantes": ["Edgar Arcia Velásquez"],
    "github": "edgararciav-cpu",
    "repositorio": "https://github.com/edgararciav-cpu/mcdi500-grupo6",
    "docente": "Dr. Omar Salinas Silva",
    "situacion": ("El NOC registró 21.542 incidencias en octubre de 2026; 588 son fallas de transmisión "
                  "(enlace de microondas, fibra propia o fibra de proveedor). Varios sitios fallaron "
                  "más de una vez en el mes, pero no hay una medición que diga cuáles concentran las fallas, "
                  "cuánto tiempo sin servicio acumulan y qué tipo de enlace las origina. Para el uptime cuentan "
                  "solo las incidencias con pérdida de servicio: criticidad P1, P2, P3 y P3E (P3 extendida, que "
                  "se trata igual que una P3)."),
    "pregunta": ("¿Qué sitios de la red concentran incidencias recurrentes de transmisión (microondas, fibra "
                 "propia FOP y fibra de proveedor), cuánto tiempo fuera de servicio acumulan y qué tipo de "
                 "enlace o proveedor las origina, para priorizar su escalamiento y evaluar cambios de "
                 "transmisión que mejoren el uptime?"),
    "objetivo_general": ("Identificar los sitios con fallas recurrentes de transmisión en octubre de 2026, "
                         "cuantificar su tiempo fuera de servicio y determinar qué tipo de enlace y qué proveedor "
                         "las originan, para priorizar su escalamiento y evaluar cambios de transmisión que "
                         "mejoren el uptime."),
    "objetivos_especificos": [
        "Cuantificar, por sitio, las fallas de transmisión con pérdida de servicio (P1, P2, P3 y P3E) y los minutos que estuvo fuera de servicio en octubre, sin contar dos veces el tiempo de incidencias que se traslapan.",
        "Identificar los sitios con fallas recurrentes (dos o más en el mes) y medir qué parte del tiempo fuera de servicio por transmisión concentran.",
        "Determinar qué tipo de enlace (microondas, fibra propia FOP, fibra de proveedor o satelital) y qué proveedor de transmisión originan más fallas y más minutos fuera de servicio.",
        "Ordenar los sitios recurrentes en una lista de prioridad para escalar al área de transmisión o al proveedor, con su enlace, su proveedor y su región.",
    ],
    "metodologia": ("Los objetivos se responden con un flujo reproducible en cuatro fases: F1 define el problema y mide "
                    "la calidad de los datos sin corregirlos; F2 limpia, transforma y arma la tabla de recurrencia por "
                    "sitio; F3 lleva ese pipeline a módulos y clases, verifica que el resultado no cambie y mide su "
                    "eficiencia; F4 visualiza y comunica los resultados."),
    "alcance": {
        "incluye": ["las 21.542 incidencias de octubre de 2026 se limpian completas",
                    "el análisis se enfoca en las 588 fallas de transmisión (Tx Propia y Tx Proveedor)",
                    "limpieza, transformación y validación (F1 y F2)"],
        "excluye": ["la decisión de cambiar la transmisión de un sitio (la toman las áreas de transmisión)",
                    "costos de enlaces o multas asociadas", "modelos predictivos (F3 y F4)",
                    "comparación con otros meses"],
    },
    "restricciones": [
        "Un solo mes de datos: la recurrencia se mide dentro de octubre y no permite analizar estacionalidad ni tendencia.",
        "El exporte no informa cuántos clientes afectó cada incidencia; solo se conoce el rango que fija la criticidad (P1 más de 50.000, P2 más de 30.000, P3 y P3E menos de 30.000).",
        "Solo cuenta el tiempo de octubre: una caída que empieza en octubre y se normaliza en noviembre aporta al uptime solo sus minutos hasta el 31-10 a las 23:59.",
        "Las incidencias aún no normalizadas no tienen fecha de cierre ni duración.",
        "El exporte no tiene una columna con el tipo de enlace; se obtiene del texto de la incidencia NOC (MW = microondas, FOP = fibra propia).",
    ],
    "supuestos": [
        "El exporte contiene todas las incidencias registradas en el mes.",
        "Una duración vacía corresponde a una incidencia aún no normalizada, no a un error de registro (se verifica en la sección 5.4).",
        "Las duraciones registradas son correctas: no hay forma de contrastarlas con otra fuente.",
        "El tipo de incidencia NOC (Tx Propia o Tx Proveedor) identifica bien las fallas de transmisión (su tamaño se mide en la sección 5.5).",
        "Solo las criticidades P1, P2, P3 y P3E corresponden a pérdida de servicio y afectan el uptime; P2 Redundancia, P2R y P4 no (definición operativa del NOC).",
    ],
    "criterios_exito": [
        "El notebook corre completo con Restart & Run All, sin errores.",
        "Cada decisión de la Fase 2 queda respaldada por una cifra del propio análisis.",
    ],
}
print("Proyecto definido:", PROYECTO["titulo"])

Proyecto definido: Fallas recurrentes de transmisión en la red de un operador móvil


La función siguiente presenta el proyecto en pantalla. Recibe el diccionario **como parámetro** y
no lee la variable global. Así se puede reutilizar y probar por separado.

In [3]:
def presentar_proyecto(config):
    '''Imprime la definición del proyecto de forma legible.

    Parámetros
    ----------
    config : dict
        Diccionario con la definición del proyecto (ver PROYECTO).
    '''
    print(config["titulo"].upper())
    print("=" * len(config["titulo"]))
    print(f"{config['asignatura']} · {config['grupo']}")
    print("Integrante:", ", ".join(config["integrantes"]), "· Docente:", config["docente"])
    print("\nSituación:\n ", config["situacion"])
    print("\nPregunta analizable:\n ", config["pregunta"])
    print("\nObjetivo general:\n ", config["objetivo_general"])
    print("\nObjetivos específicos:")
    for i, objetivo in enumerate(config["objetivos_especificos"], start=1):
        print(f"  {i}. {objetivo}")
    print("\nMetodología:\n ", config["metodologia"])

presentar_proyecto(PROYECTO)

FALLAS RECURRENTES DE TRANSMISIÓN EN LA RED DE UN OPERADOR MÓVIL
MCDI500 · Programación para la Ciencia de Datos · Grupo 6 (trabajo individual, informado al docente)
Integrante: Edgar Arcia Velásquez · Docente: Dr. Omar Salinas Silva

Situación:
  El NOC registró 21.542 incidencias en octubre de 2026; 588 son fallas de transmisión (enlace de microondas, fibra propia o fibra de proveedor). Varios sitios fallaron más de una vez en el mes, pero no hay una medición que diga cuáles concentran las fallas, cuánto tiempo sin servicio acumulan y qué tipo de enlace las origina. Para el uptime cuentan solo las incidencias con pérdida de servicio: criticidad P1, P2, P3 y P3E (P3 extendida, que se trata igual que una P3).

Pregunta analizable:
  ¿Qué sitios de la red concentran incidencias recurrentes de transmisión (microondas, fibra propia FOP y fibra de proveedor), cuánto tiempo fuera de servicio acumulan y qué tipo de enlace o proveedor las origina, para priorizar su escalamiento y evaluar camb

El alcance, las restricciones y los supuestos van aparte, porque cumplen otra función: delimitan lo
que **no** se hará y las condiciones bajo las que los resultados son válidos. Hay que distinguirlos:

- **Alcance:** lo que decide el proyecto, por ejemplo "no se comparan otros meses".
- **Restricción:** una condición impuesta por los datos, por ejemplo "solo hay un mes".

In [4]:
for titulo, lista in [("Alcance · incluye", PROYECTO["alcance"]["incluye"]),
                      ("Alcance · excluye", PROYECTO["alcance"]["excluye"]),
                      ("Restricciones", PROYECTO["restricciones"]),
                      ("Supuestos", PROYECTO["supuestos"])]:
    print(titulo)
    for elemento in lista:
        print("  -", elemento)

Alcance · incluye
  - las 21.542 incidencias de octubre de 2026 se limpian completas
  - el análisis se enfoca en las 588 fallas de transmisión (Tx Propia y Tx Proveedor)
  - limpieza, transformación y validación (F1 y F2)
Alcance · excluye
  - la decisión de cambiar la transmisión de un sitio (la toman las áreas de transmisión)
  - costos de enlaces o multas asociadas
  - modelos predictivos (F3 y F4)
  - comparación con otros meses
Restricciones
  - Un solo mes de datos: la recurrencia se mide dentro de octubre y no permite analizar estacionalidad ni tendencia.
  - El exporte no informa cuántos clientes afectó cada incidencia; solo se conoce el rango que fija la criticidad (P1 más de 50.000, P2 más de 30.000, P3 y P3E menos de 30.000).
  - Solo cuenta el tiempo de octubre: una caída que empieza en octubre y se normaliza en noviembre aporta al uptime solo sus minutos hasta el 31-10 a las 23:59.
  - Las incidencias aún no normalizadas no tienen fecha de cierre ni duración.
  - El expor

## 2. Verificación del entorno

**Qué hace este paso.** Comprueba con evidencia, y no por confianza, que el notebook corre con el
**intérprete del proyecto** (`.venv`) y que cada librería del proyecto instalada tiene la **misma versión**
que declara `requirements.txt`. Si una versión no coincide, lo informa y el notebook sigue, para que se
pueda revisar el resto; si el kernel no es el del proyecto, se detiene.

`requirements.txt` declara solo las librerías que el proyecto importa (pandas, NumPy, scikit-learn y
matplotlib) y las que ejecutan los notebooks (JupyterLab e ipykernel), cada una con versión fija. Las
dependencias de esas librerías las resuelve pip según el sistema: así el mismo archivo se instala en
Windows, macOS y Linux (en la Sumativa 1 el archivo era un volcado de 105 paquetes con uno exclusivo de
Windows).

**Por qué es lo primero.** Si el kernel no es el del proyecto, todo lo que viene después puede dar
otro resultado en otro equipo. Es el origen de casi todos los "a mí me funciona" (lámina 34 de la
presentación de la Semana 1).

In [5]:
def localizar_raiz(inicio=None):
    '''Devuelve la raíz del proyecto: la primera carpeta hacia arriba que contiene requirements.txt.

    Jupyter abre el notebook en su propia carpeta (F1/), no en la raíz; esta función permite usar
    rutas relativas a la raíz desde cualquier notebook.
    '''
    carpeta = Path(inicio or Path.cwd()).resolve()
    for candidata in [carpeta, *carpeta.parents]:
        if (candidata / "requirements.txt").is_file():
            return candidata
    raise FileNotFoundError("No encontré requirements.txt en esta carpeta ni en las superiores.")


def leer_requirements(ruta):
    '''Lee requirements.txt y devuelve {paquete: versión fijada}.'''
    versiones = {}
    for linea in Path(ruta).read_text(encoding="utf-8").splitlines():
        linea = linea.split("#", 1)[0]          # los comentarios no declaran paquetes
        if "==" in linea:
            paquete, ver = linea.strip().split("==", 1)
            versiones[paquete.lower()] = ver
    return versiones


RAIZ = localizar_raiz()
DECLARADAS = leer_requirements(RAIZ / "requirements.txt")

# Nombre en PyPI (requirements) -> nombre con que se importa
CLAVE = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "scikit-learn": "sklearn",
         "jupyterlab": None, "ipykernel": None}

def verificar_entorno(declaradas, claves):
    '''Compara la versión instalada de cada paquete clave con la declarada. Devuelve una tabla.'''
    filas = []
    for paquete, modulo in claves.items():
        try:
            instalada = version(paquete)
        except PackageNotFoundError:
            instalada = None
        if modulo:                      # además de estar instalado, debe poder importarse
            importlib.import_module(modulo)
        filas.append({"paquete": paquete, "declarada": declaradas.get(paquete),
                      "instalada": instalada, "coincide": instalada == declaradas.get(paquete)})
    return pd.DataFrame(filas)


print("Raíz del proyecto :", RAIZ)
print("Intérprete        :", sys.executable)
print("Python            :", platform.python_version(), "·", platform.system(), platform.release())
print("Paquetes fijados en requirements.txt:", len(DECLARADAS))
entorno = verificar_entorno(DECLARADAS, CLAVE)
entorno

Raíz del proyecto : C:\Users\edgar\MCDI500\mcdi500-grupo6
Intérprete        : C:\Users\edgar\MCDI500\mcdi500-grupo6\.venv\Scripts\python.exe
Python            : 3.14.6 · Windows 11
Paquetes fijados en requirements.txt: 6


,paquete,declarada,instalada,coincide
0,numpy,2.5.4,2.5.4,True
1,pandas,3.1.1,3.1.1,True
2,matplotlib,3.11.2,3.11.2,True
3,scikit-learn,1.9.1,1.9.1,True
4,jupyterlab,4.6.6,4.6.6,True
5,ipykernel,7.4.0,7.4.0,True


In [6]:
assert ".venv" in sys.executable, "El kernel no es el del proyecto: Kernel > Change Kernel > Python (mcdi500)"
assert set(CLAVE) == set(DECLARADAS), f"requirements.txt y la lista de verificación no declaran lo mismo: {set(CLAVE) ^ set(DECLARADAS)}"
distintas = entorno[~entorno["coincide"]]
if distintas.empty:
    print("[OK] El notebook corre en .venv y las", len(entorno), "librerías de requirements.txt tienen la versión declarada")
else:                                   # se informa y se sigue: el resto del notebook se puede revisar igual
    print("[AVISO] Versiones distintas a las declaradas (las cifras podrían variar):")
    print(distintas[["paquete", "declarada", "instalada"]].to_string(index=False))
    print("Para alinearlas: .venv/Scripts/python.exe -m pip install -r requirements.txt")

[OK] El notebook corre en .venv y las 6 librerías de requirements.txt tienen la versión declarada


**Cómo leer este resultado.**

| Si se ve | Qué significa | Qué hacer |
|---|---|---|
| El intérprete no contiene `.venv` | El kernel no es el del proyecto | Kernel → Change Kernel → *Python (mcdi500)* |
| `coincide` en False | La versión instalada no es la declarada | `pip install -r requirements.txt` |
| Error al importar | La librería no está instalada en `.venv` | Instalar desde `requirements.txt` |

## 3. Estructura del repositorio y archivos de reproducibilidad

**Qué hace este paso.** Comprueba que existan las carpetas del proyecto y que `.gitignore` y
`requirements.txt` cumplan su función. No los crea: se crearon y versionaron antes, en los commits
`c7469f7` y `db429ec`, y aquí se verifica que siguen correctos.

**Decisión documentada.** Los datos son comunes a todas las fases (`data/` en la raíz) y cada fase
tiene su carpeta con su notebook, igual que el proyecto modelo del curso.

In [7]:
CARPETAS = ["F1", "F2", "F3", "F4", "data/raw", "data/processed", "docs", "src", "scripts"]
DIR_CRUDO, DIR_PROCESADO = RAIZ / "data" / "raw", RAIZ / "data" / "processed"
DIR_DOCS, DIR_SRC = RAIZ / "docs", RAIZ / "src"

estructura = pd.DataFrame([{"carpeta": c, "existe": (RAIZ / c).is_dir(),
                            "archivos": sum(1 for p in (RAIZ / c).glob("*") if p.is_file() and p.name != ".gitkeep")}
                           for c in CARPETAS])
gitignore = (RAIZ / ".gitignore").read_text(encoding="utf-8")
print(".gitignore excluye .venv/            :", ".venv/" in gitignore)
print(".gitignore excluye el exporte original:", "Export_Incidencias" in gitignore)
lineas_req = [l.split("#", 1)[0].strip() for l in (RAIZ / "requirements.txt").read_text(encoding="utf-8").splitlines()]
lineas_req = [l for l in lineas_req if l]
print("requirements.txt fija versiones con '==':", all("==" in l for l in lineas_req), f"({len(lineas_req)} librerías)")
estructura

.gitignore excluye .venv/            : True
.gitignore excluye el exporte original: True
requirements.txt fija versiones con '==': True (6 librerías)


,carpeta,existe,archivos
0,F1,True,2
1,F2,True,2
2,F3,True,2
3,F4,True,1
4,data/raw,True,1
5,data/processed,True,5
6,docs,True,13
7,src,True,7
8,scripts,True,2


## 4. Módulos: funciones reutilizables en `src/`

**Qué hace este paso.** Importa el módulo propio `src/utilidades.py` y lo prueba.

**Por qué un módulo.** La Unidad 1 pide funciones, excepciones **y módulos**. El notebook documenta
y comunica; el módulo permite que la misma función se use en F2, F3 y F4 sin copiarla. Contiene:

| Elemento | Para qué |
|---|---|
| `reparar_texto` | Corrige tildes rotas por doble codificación (`EnergÃ­a` → `Energía`) |
| `normalizar_nombre`, `normalizar_columnas` | Nombres de columna en snake_case, sin tildes ni repetidos |
| `duracion_a_minutos` | Convierte `h:mm:ss` a minutos y rechaza formatos inválidos |
| `ConjuntoDatos` (clase) | Carga el archivo siempre con el mismo separador y codificación |
| `ProyectoError` (excepción propia) | Distingue un error de los datos de un error de pandas |

In [8]:
if str(DIR_SRC) not in sys.path:
    sys.path.insert(0, str(DIR_SRC))
import utilidades
importlib.reload(utilidades)       # toma la última versión del archivo si se editó
print("Módulo importado desde:", Path(utilidades.__file__).relative_to(RAIZ))

Módulo importado desde: src\utilidades.py


### 4.1 Pruebas del módulo: casos normales, límite y excepciones

`assert` detiene el notebook si una condición no se cumple y deja constancia en la salida. Son los
mismos tres tipos de prueba que la rúbrica exige en la Fase 2 (criterio 11).

In [9]:
# --- Casos normales ---
assert utilidades.reparar_texto("EnergÃ­a") == "Energía"
assert utilidades.normalizar_nombre("Tipo Incidencia NOC") == "tipo_incidencia_noc"
assert utilidades.normalizar_nombre("Tiempo de duraciÃ³n de la incidencia") == "tiempo_de_duracion_de_la_incidencia"
assert utilidades.duracion_a_minutos("1:30:00") == 90
assert utilidades.duracion_a_minutos("158:11:17") == 158 * 60 + 11 + 17 / 60
print("[OK] Casos normales")

# --- Casos límite ---
assert utilidades.reparar_texto("Cerrada") == "Cerrada"                  # texto sano: no se altera
assert utilidades.reparar_texto(None) is None                            # no es texto: se devuelve igual
assert utilidades.normalizar_nombre("</div>ï»¿TT") == "tt"               # la cabecera rota del exporte
assert utilidades.duracion_a_minutos("0:00:00") == 0
assert np.isnan(utilidades.duracion_a_minutos(""))                       # vacía = incidencia abierta
print("[OK] Casos límite")

# --- Excepciones: entradas inválidas deben fallar con el error propio ---
for entrada_invalida in ["1:30", "a:b:c", "1:75:00"]:
    try:
        utilidades.duracion_a_minutos(entrada_invalida)
        raise AssertionError(f"debió fallar con {entrada_invalida!r}")
    except utilidades.ProyectoError:
        pass
try:
    utilidades.normalizar_columnas(["Estado", "estado"])
    raise AssertionError("debió detectar nombres repetidos")
except utilidades.ProyectoError:
    pass
print("[OK] Excepciones: formatos inválidos y nombres repetidos se rechazan con ProyectoError")

[OK] Casos normales
[OK] Casos límite
[OK] Excepciones: formatos inválidos y nombres repetidos se rechazan con ProyectoError


## 5. Selección y ficha del conjunto de datos

**Qué hace este paso.** Documenta de dónde vienen los datos y los evalúa contra los requisitos del
curso, con las cifras medidas sobre el archivo real.

In [10]:
FICHA = {
    "titulo": "Incidencias de red registradas por el NOC · octubre de 2026",
    "origen": "Exporte interno del sistema de tickets del NOC de WOM (operador móvil, Chile)",
    "permiso": "Los sitios se reemplazaron por códigos y el texto libre se omitió; el docente está informado",
    "periodo": "2026-10-01 a 2026-10-31 (cierres hasta 2026-11-10)",
    "anonimizacion": "scripts/anonimizar_incidencias.py: personas, tickets de proveedor y códigos de servicio TX reemplazados por códigos; los sitios van con un código (SIT_0001, ...) y el texto libre (Titulo y Detalle) se omite",
    "archivo": "data/raw/incidencias_oct2026_anon.csv",
    "separador": ";",
    "codificacion": "utf-8-sig",
}
DATOS = utilidades.ConjuntoDatos(RAIZ / FICHA["archivo"], sep=FICHA["separador"], encoding=FICHA["codificacion"])
df_crudo = DATOS.cargar()                     # lectura tal cual: los tipos los infiere pandas
df_texto = DATOS.cargar(como_texto=True)      # misma lectura, todo como texto, para auditar vacíos reales
print(DATOS)
print("Forma:", df_crudo.shape[0], "filas ×", df_crudo.shape[1], "columnas")

ConjuntoDatos('incidencias_oct2026_anon.csv', sep=';', encoding='utf-8-sig')
Forma: 21542 filas × 49 columnas


### 5.1 Evaluación contra los requisitos del curso

Requisitos **obligatorios** (descalifican si faltan) y **avisos** (lo que falte se declara como
restricción), según el anuncio del docente del 03-11 y `validar_dataset.py`.

In [11]:
def evaluar_requisitos(df, ficha):
    '''Compara el conjunto con los requisitos del curso. Devuelve una tabla con el veredicto.'''
    pct_nulos = df.isna().mean() * 100
    fechas = [c for c in df.columns if "fecha" in utilidades.normalizar_nombre(c)]
    filas = [
        ("Al menos 2.000 filas", "obligatorio", len(df) >= 2000, f"{len(df):,} filas".replace(",", ".")),
        ("Al menos 12 columnas", "obligatorio", df.shape[1] >= 12, f"{df.shape[1]} columnas"),
        ("Al menos una variable numérica", "obligatorio", df.select_dtypes("number").shape[1] >= 1,
         f"{df.select_dtypes('number').shape[1]} numéricas al leer"),
        ("Al menos una variable categórica", "obligatorio", df.select_dtypes(exclude="number").shape[1] >= 1,
         f"{df.select_dtypes(exclude='number').shape[1]} de texto o categoría"),
        ("Presencia de valores faltantes", "aviso", pct_nulos.max() > 0, f"máximo {pct_nulos.max():.1f} %"),
        ("Variable de fecha", "aviso", len(fechas) > 0, f"{len(fechas)} columnas de fecha"),
        ("Archivo bajo 100 MB", "aviso", DATOS.ruta.stat().st_size < 100e6, f"{DATOS.ruta.stat().st_size / 1e6:.1f} MB"),
        ("Datos de los últimos 5 años", "obligatorio", True, ficha["periodo"]),
        ("Sin datos personales identificables", "obligatorio", True, "anonimizado (ver bitácora, paso 4)"),
    ]
    return pd.DataFrame(filas, columns=["criterio", "tipo", "cumple", "evidencia"])

evaluacion = evaluar_requisitos(df_crudo, FICHA)
assert evaluacion.loc[evaluacion["tipo"] == "obligatorio", "cumple"].all(), "Falla un requisito obligatorio"
evaluacion

,criterio,tipo,cumple,evidencia
0,Al menos 2.000 filas,obligatorio,True,21.542 filas
1,Al menos 12 columnas,obligatorio,True,49 columnas
2,Al menos una variable numérica,obligatorio,True,2 numéricas al leer
3,Al menos una variable categórica,obligatorio,True,47 de texto o categoría
4,Presencia de valores faltantes,aviso,True,máximo 100.0 %
5,Variable de fecha,aviso,True,4 columnas de fecha
6,Archivo bajo 100 MB,aviso,True,5.7 MB
7,Datos de los últimos 5 años,obligatorio,True,2026-10-01 a 2026-10-31 (cierres hasta 2026-11...
8,Sin datos personales identificables,obligatorio,True,"anonimizado (ver bitácora, paso 4)"


### 5.2 Diccionario de variables

El diccionario declara el **rol analítico** de cada variable. El tipo que asigna pandas no basta:
`Q Network Element` es un número, pero `TT` es un identificador aunque también tenga dígitos. El
rol decide el tratamiento de la Fase 2.

El nombre original se conserva tal como viene en el exporte, con su codificación rota, y al lado va
el nombre normalizado que usará la Fase 2.

In [12]:
ROLES = {
    "TT": ("identificador", "Número de ticket de la incidencia (no es único: un ticket puede abarcar varias filas)"),
    "Fecha de Inicio de incidencia": ("fecha", "Inicio de la incidencia (dd-mm-aaaa H:MM)"),
    "Fecha de Inicio ticket": ("fecha", "Apertura del ticket"),
    "Comuna": ("alta cardinalidad", "Comuna del elemento afectado"),
    "Estado": ("nominal", "Cerrada, Pendiente Automático, Abierta, Cancelada"),
    "Tipo Incidencia NOC": ("nominal", "Tipo de falla según el NOC (Energía, Tx, RF, Hardware, ...)"),
    "Incidencia NOC": ("nominal", "Subtipo de falla"),
    "Criticidad": ("ordinal", "P1 más de 50.000 clientes afectados, P2 más de 30.000, P3 y P3E menos de 30.000 (P3E: P3 extendida, se trata igual que una P3). P1, P2, P3 y P3E tienen pérdida de servicio; P2 Redundancia y P2R no (la redundancia está activa)"),
    "Detalle": ("texto", "Texto libre del operador; la anonimización lo reemplaza por [texto omitido]"),
    "Impacto": ("ordinal", "Sin Impacto < Localizado < Regional < Amplio"),
    "Tiempo de Respuesta": ("discreta", "Tiempo comprometido de respuesta en horas (0, 2, 4, 24, 48)"),
    "Titulo": ("texto", "Título del ticket; la anonimización lo reemplaza por [texto omitido]"),
    "Q Network Element": ("discreta", "Cantidad de sitios afectados por la incidencia"),
    "Network Element": ("identificador", "Código del sitio afectado (SIT_); si son varios, separados por dos espacios"),
    "Proveedor": ("nominal", "Proveedor asociado a la falla (eléctrico, TX, propietario)"),
    "Ticket de proveedor": ("identificador", "Ticket del proveedor (código TKP_ o marcador como N/A)"),
    "Incident Manager": ("nominal", "Responsable de la incidencia (código IM_)"),
    "Creador de TT": ("nominal", "Operador que creó el ticket (código OP_)"),
    "Fecha Cierre de incidencia": ("fecha", "Normalización de la incidencia"),
    "Fecha de Cierre de ticket": ("fecha", "Cierre del ticket"),
    "Tiempo de duración de la incidencia": ("continua", "Duración h:mm:ss; vacía si la incidencia no se ha normalizado"),
    "Tiempo de duración del ticket": ("continua", "Duración del ticket h:mm:ss"),
    "Código de Servicio": ("identificador", "Código de servicio (suministro eléctrico, no es RUT)"),
    "Tramo Afectado": ("texto", "Tramo de transmisión afectado; casi siempre NA o no aplica"),
    "Comunas": ("alta cardinalidad", "Comunas afectadas (puede listar varias)"),
    "Regiones": ("nominal", "Regiones afectadas en números romanos (puede listar varias)"),
    "Causa Raiz NOC": ("nominal", "Causa raíz según el NOC"),
    "Tipo Incidencia GMI": ("nominal", "Tipo según el área GMI"),
    "Incidenca GMI": ("nominal", "Subtipo según GMI (el nombre de la columna viene con error de tipeo)"),
    "Causa Raiz GMI": ("nominal", "Causa raíz según GMI"),
    "Horario Posible Degradación": ("vacía", "Sin datos en el exporte"),
    "ETR": ("fecha", "Hora estimada de restauración"),
    "Gestionable": ("binaria", "SI / NO"),
    "Gestor Uptime": ("binaria", "SI / NO"),
}
for i in range(1, 6):
    ROLES[f"Proveedor TX {i}"] = ("nominal", f"Proveedor del enlace de transmisión {i}")
    ROLES[f"Código servicio TX {i}"] = ("identificador", f"Código del servicio TX {i} (TXS_)")
    ROLES[f"Descripción TX {i}"] = ("texto", f"Descripción del enlace TX {i}")

def construir_diccionario(df, roles):
    '''Une el rol declarado de cada columna con lo que se mide en el archivo.'''
    filas = []
    for original in df.columns:
        legible = utilidades.reparar_texto(original)
        rol, descripcion = roles.get(legible, ("sin clasificar", ""))
        serie = df[original]
        filas.append({"variable": utilidades.normalizar_nombre(original), "nombre_original": original,
                      "rol_analitico": rol, "descripcion": descripcion, "dtype": str(serie.dtype),
                      "n_unicos": int(serie.nunique()), "pct_nulos": round(serie.isna().mean() * 100, 2)})
    return pd.DataFrame(filas)

diccionario = construir_diccionario(df_crudo, ROLES)
assert (diccionario["rol_analitico"] != "sin clasificar").all(), "Hay columnas sin rol declarado"
print("Variables por rol:", diccionario["rol_analitico"].value_counts().to_dict())
diccionario

Variables por rol: {'nominal': 16, 'identificador': 9, 'texto': 8, 'fecha': 5, 'alta cardinalidad': 2, 'ordinal': 2, 'discreta': 2, 'continua': 2, 'binaria': 2, 'vacía': 1}


,variable,nombre_original,rol_analitico,descripcion,dtype,n_unicos,pct_nulos
0,tt,</div>ï»¿TT,identificador,Número de ticket de la incidencia (no es único...,str,19915,0.00
1,fecha_de_inicio_de_incidencia,Fecha de Inicio de incidencia,fecha,Inicio de la incidencia (dd-mm-aaaa H:MM),str,14997,0.00
2,fecha_de_inicio_ticket,Fecha de Inicio ticket,fecha,Apertura del ticket,str,4598,0.00
3,comuna,Comuna,alta cardinalidad,Comuna del elemento afectado,str,335,1.59
4,estado,Estado,nominal,"Cerrada, Pendiente Automático, Abierta, Cancelada",str,4,0.00
5,tipo_incidencia_noc,Tipo Incidencia NOC,nominal,"Tipo de falla según el NOC (Energía, Tx, RF, H...",str,14,0.00
6,incidencia_noc,Incidencia NOC,nominal,Subtipo de falla,str,30,0.00
7,criticidad,Criticidad,ordinal,"P1 más de 50.000 clientes afectados, P2 más de...",str,7,0.00
8,detalle,Detalle,texto,Texto libre del operador; la anonimización lo ...,str,1,0.00
9,impacto,Impacto,ordinal,Sin Impacto < Localizado < Regional < Amplio,str,4,0.00


### 5.3 Reconocimiento inicial (sin modificar nada)

**Qué hace este paso.** Mide los problemas que trae el archivo para que la Fase 2 decida con cifras.
Aquí **no se corrige nada**: solo se cuenta.

In [13]:
def medir_problemas(df_txt):
    '''Cuenta los problemas de calidad visibles en el archivo leído como texto.'''
    celdas_rotas = df_txt.apply(lambda col: col.str.contains("Ã|Â", regex=True)).sum().sum()
    col_tramo = [c for c in df_txt.columns if utilidades.reparar_texto(c) == "Tramo Afectado"][0]
    variantes_na = df_txt[col_tramo][df_txt[col_tramo].str.strip().str.lower().isin(["na", "n/a", "no aplica"])].unique()
    col_tt = df_txt.columns[0]
    col_dur = [c for c in df_txt.columns if utilidades.reparar_texto(c) == "Tiempo de duración de la incidencia"][0]
    dur = df_txt[col_dur].str.strip()
    dur_invalida = dur.ne("") & ~dur.str.fullmatch(r"\d+:\d{2}:\d{2}")
    return pd.DataFrame([
        ("Duraciones con formato inválido (por ejemplo, negativas)", int(dur_invalida.sum())),
        ("Celdas con tildes rotas (Ã, Â)", int(celdas_rotas)),
        ("Formas de escribir 'no aplica' en Tramo Afectado", len(variantes_na)),
        ("Filas cuyo TT se repite en otra fila", int(df_txt[col_tt].duplicated(keep=False).sum())),
        ("Filas duplicadas exactas", int(df_txt.duplicated().sum())),
        ("Columnas con más de 60 % vacío", int((df_txt.eq("").mean() > 0.6).sum())),
    ], columns=["problema", "cantidad"])

problemas = medir_problemas(df_texto)
problemas

,problema,cantidad
0,"Duraciones con formato inválido (por ejemplo, ...",2
1,"Celdas con tildes rotas (Ã, Â)",43634
2,Formas de escribir 'no aplica' en Tramo Afectado,5
3,Filas cuyo TT se repite en otra fila,2708
4,Filas duplicadas exactas,0
5,Columnas con más de 60 % vacío,26


### 5.4 Verificación del supuesto sobre las duraciones vacías

El supuesto 2 de la sección 1 dice que una duración vacía es una incidencia **aún no normalizada**.
Se verifica cruzando el estado con la presencia de duración.

In [14]:
col_estado = "Estado"
col_duracion = [c for c in df_texto.columns if utilidades.reparar_texto(c) == "Tiempo de duración de la incidencia"][0]
duracion_vacia = df_texto[col_duracion].str.strip().eq("")
cruce = pd.crosstab(df_texto[col_estado], duracion_vacia.map({True: "duración vacía", False: "con duración"}),
                    margins=True, margins_name="Total").rename_axis(columns="Duración")
print(f"Duraciones vacías: {duracion_vacia.sum():,} de {len(df_texto):,} ({duracion_vacia.mean():.1%})".replace(",", "."))
assert set(df_texto.loc[duracion_vacia, col_estado]) <= {"Abierta", "Pendiente Automatico"}, "Hay vacíos en incidencias cerradas"
print("[OK] Supuesto verificado: todas las duraciones vacías son incidencias Abiertas o Pendientes (no normalizadas)")
cruce

Duraciones vacías: 5.318 de 21.542 (24.7%)
[OK] Supuesto verificado: todas las duraciones vacías son incidencias Abiertas o Pendientes (no normalizadas)


Duración,con duración,duración vacía,Total
Estado,,,
Abierta,0,313,313
Cancelada,52,0,52
Cerrada,16172,0,16172
Pendiente Automatico,0,5005,5005
Total,16224,5318,21542


> **Consecuencia para la Fase 2.** El vacío **no ocurre al azar**: depende del estado. Rellenarlo con
> la media o la mediana inventaría una hora de cierre que no existe. La Fase 2 igual compara los
> métodos, porque la rúbrica lo pide, pero la decisión debe considerar este hecho.

### 5.5 Tamaño del foco de análisis: fallas de transmisión

**Qué hace este paso.** Cuenta cuántas incidencias son de transmisión, de qué subtipo y cuántos
sitios fallan más de una vez.

Una incidencia puede afectar a varios sitios: el exporte los escribe en la misma celda de
`Network Element`, separados por **dos espacios** (en el exporte original un espacio simple es parte del nombre de
algunos sitios; en el archivo anonimizado cada sitio tiene un código `SIT_` sin espacios). La recurrencia se cuenta **por sitio**, con `utilidades.separar_sitios`, y se
comprueba que la cantidad separada coincide con `Q Network Element`. Sirve para confirmar que la pregunta tiene datos suficientes
para responderse. Igual que en 5.3, aquí **solo se cuenta**: la clasificación por tipo de enlace y
la tabla de recurrencia con minutos se construyen en la Fase 2, con los datos ya limpios.

In [15]:
TIPOS_TX = ["Tx Propia", "Tx Proveedor"]
es_tx = df_texto["Tipo Incidencia NOC"].isin(TIPOS_TX)
tx = df_texto[es_tx]
print(f"Fallas de transmisión: {len(tx)} de {len(df_texto):,} incidencias ({es_tx.mean():.1%})".replace(",", "."))
print("Por tipo:", tx["Tipo Incidencia NOC"].value_counts().to_dict())

CRITICIDAD_UPTIME = ["P1", "P2", "P3", "P3E"]          # pérdida de servicio: afectan el uptime
afecta_uptime = tx["Criticidad"].isin(CRITICIDAD_UPTIME)
print(f"Con pérdida de servicio (P1, P2, P3, P3E): {afecta_uptime.sum()} de {len(tx)} fallas de transmisión")

sitios_por_fila = df_texto["Network Element"].map(utilidades.separar_sitios)
assert (sitios_por_fila.map(len) == df_texto["Q Network Element"].astype(int)).all(), "La separación no coincide con Q Network Element"
print(f"[OK] Separar por dos espacios coincide con Q Network Element en las {len(df_texto):,} filas".replace(",", "."))
print(f"Incidencias que afectan a más de un sitio: {int((sitios_por_fila.map(len) > 1).sum()):,}".replace(",", "."))

def medir_recurrencia(celdas):
    '''Cuenta cuántas veces falla cada sitio (separando las celdas con varios sitios) y resume la recurrencia.'''
    conteo = celdas.map(utilidades.separar_sitios).explode().value_counts()
    return {"Sitios con al menos 1 falla": len(conteo),
            "Sitios con 2 o más fallas en el mes": int((conteo >= 2).sum()),
            "Sitios con 3 o más fallas en el mes": int((conteo >= 3).sum()),
            "Máximo de fallas de un mismo sitio": int(conteo.max())}

fallas_por_elemento = tx["Network Element"].map(utilidades.separar_sitios).explode().value_counts()
fallas_uptime_por_elemento = tx.loc[afecta_uptime, "Network Element"].map(utilidades.separar_sitios).explode().value_counts()
recurrencia = pd.DataFrame({"todas las fallas TX": medir_recurrencia(tx["Network Element"]),
                            "solo pérdida de servicio": medir_recurrencia(tx.loc[afecta_uptime, "Network Element"])})
assert (fallas_uptime_por_elemento >= 2).any(), "No hay recurrencia con pérdida de servicio: la pregunta no tendría datos"
print("[OK] Hay sitios con pérdidas de servicio repetidas: la pregunta tiene datos para responderse")
print("Sitios con más pérdidas de servicio:", fallas_uptime_por_elemento.head(5).to_dict())
recurrencia

Fallas de transmisión: 588 de 21.542 incidencias (2.7%)
Por tipo: {'Tx Proveedor': 333, 'Tx Propia': 255}
Con pérdida de servicio (P1, P2, P3, P3E): 264 de 588 fallas de transmisión
[OK] Separar por dos espacios coincide con Q Network Element en las 21.542 filas
Incidencias que afectan a más de un sitio: 1.098
[OK] Hay sitios con pérdidas de servicio repetidas: la pregunta tiene datos para responderse
Sitios con más pérdidas de servicio: {'SIT_0535': 11, 'SIT_0751': 4, 'SIT_1088': 4, 'SIT_1452': 4, 'SIT_0533': 3}


,todas las fallas TX,solo pérdida de servicio
Sitios con al menos 1 falla,721,561
Sitios con 2 o más fallas en el mes,366,151
Sitios con 3 o más fallas en el mes,158,21
Máximo de fallas de un mismo sitio,19,11


> **Por qué se separan las dos columnas.** Una falla de transmisión sin pérdida de servicio (por
> ejemplo, una pérdida de redundancia en P2 Redundancia o P4) no baja el uptime. Si se cuentan todas,
> aparecen arriba sitios que fallan seguido pero no quedan fuera de servicio. Por eso la
> tabla de recurrencia de la Fase 2 se construye con la columna de la derecha.

In [16]:
# Mediana de duración por criticidad (solo filas con duración válida): la P3E es la que pasa más tiempo fuera
col_dur = [c for c in df_texto.columns if utilidades.reparar_texto(c) == "Tiempo de duración de la incidencia"][0]
dur_txt = df_texto[col_dur].str.strip()
valida = dur_txt.str.fullmatch(r"\d+:\d{2}:\d{2}")
minutos = dur_txt[valida].map(utilidades.duracion_a_minutos)
(minutos.groupby(df_texto.loc[valida, "Criticidad"].map(utilidades.reparar_texto))
        .agg(incidencias="size", mediana_min="median").round(0).sort_values("mediana_min", ascending=False))

,incidencias,mediana_min
Criticidad,,
P3E,674,3188.0
P2 Redundancia,144,462.0
P3,1649,247.0
P2R Degradación,1,220.0
P2,13,125.0
P1,1,105.0
P4,13740,49.0


In [17]:
# Subtipos de falla: de este texto sale, en la Fase 2, el tipo de enlace (MW, FOP o fibra de proveedor)
pd.crosstab(tx["Incidencia NOC"].map(utilidades.reparar_texto),
            [tx["Tipo Incidencia NOC"], afecta_uptime.map({True: "con pérdida", False: "sin pérdida"})],
            margins=True, margins_name="Total").rename_axis(index="Incidencia NOC", columns=["Tipo", "Servicio"])

Tipo                                                 Tx Propia             Tx Proveedor             Total
Servicio                                           con pérdida sin pérdida  con pérdida sin pérdida      
Incidencia NOC                                                                                           
Atenuación en Incidencia                                     0           1            0           0     1
Corte de Fibra con afectación de servicio total              0           0          133          81   214
FOP Corte de Fibra con afectación de servicio t...          37          35            0           0    72
FOP Pérdida de Redundancia                                   3          29            0           0    32
FOP Pérdida de Redundancia con degradación                   0           3            0           0     3
Falla Modem Satelital (Subtel)                               0           0           10           5    15
MW Fading, Falla de negociación, Config. Pérdid...           6           6            0           0    12
MW Pérdida de enlace                                        70          65            0           0   135
Pérdida de Redundancia                                       0           0            5          95   100
Pérdida de Redundancia con degradación                       0           0            0           4     4
Total                                                      116         139          148         185   588

## 6. Control de versiones

**Qué hace este paso.** Consulta Git desde el notebook: la identidad configurada, la rama y el
historial. No ejecuta commits; los commits se hacen desde la terminal, con un mensaje que explica el
cambio.

In [18]:
def git(*argumentos):
    '''Ejecuta un comando de Git en la raíz del proyecto. Devuelve (éxito, salida).'''
    r = subprocess.run(["git", *argumentos], cwd=RAIZ, capture_output=True, text=True, encoding="utf-8")
    return r.returncode == 0, (r.stdout or r.stderr).strip()

_, nombre = git("config", "user.name")
_, correo = git("config", "user.email")
_, rama = git("rev-parse", "--abbrev-ref", "HEAD")
_, historial = git("log", "--format=%h | %ad | %an | %s", "--date=format:%d-%m %H:%M")
print("Identidad de Git :", nombre, f"<{correo}>")
print("Rama actual      :", rama)
print("Historial:")
print(historial)

Identidad de Git : Edgar Arcia <edgararciav@gmail.com>
Rama actual      : f3-documentacion
Historial:
c25abe5 | 16-11 22:16 | Edgar Arcia | docs: agrega .mailmap para que Git muestre una sola identidad del autor
8735fe3 | 16-11 21:29 | Edgar Arcia | docs: registra en la bitácora los merges de los pull request de la Fase 3
9933b1d | 16-11 21:28 | Edgar Arcia | Merge pull request #3 from edgararciav-cpu/f3-nucleo
84ea005 | 16-11 21:11 | Edgar Arcia | Merge pull request #2 from edgararciav-cpu/f3-confidencialidad
b0c2b7c | 16-11 21:10 | Edgar Arcia | Merge pull request #1 from edgararciav-cpu/f3-dependencias
135fa4b | 16-11 20:54 | Edgar Arcia | docs: actualiza los hashes de commits citados en la bitácora, los metadatos y los notebooks
ce72af4 | 16-11 20:39 | Edgar Arcia | feat: notebook F3 ejecutado con clases, equivalencia, eficiencia, pruebas y contrato con la Fase 4
64ea39b | 16-11 20:28 | Edgar Arcia | docs: cifras de eficiencia como rangos, porque varían entre ejecuciones
f2aaab5 | 

| Prefijo | Cuándo se usa | Ejemplo del historial |
|---|---|---|
| `docs:` | documentación | docs: crea estructura inicial del proyecto |
| `data:` | datos | data: incorpora incidencias de octubre 2026 anonimizadas en raw |
| `feat:` | algo nuevo que hace funcionar el proyecto | feat: define el entorno reproducible con requirements.txt |
| `fix:` | corrección | — |
| `test:` | pruebas | — |

## 7. Validación técnica de la fase

La función reúne las comprobaciones que la Fase 1 debe dejar cumplidas. Si alguna falla, el
notebook se detiene con un error que dice cuál.

In [19]:
def validar_fase1(raiz, config, evaluacion, diccionario):
    '''Comprueba que la Fase 1 deja lo que exige la rúbrica. Lanza AssertionError si algo falla.'''
    comprobaciones = [
        ("Problema con pregunta, objetivo general y 3 a 5 objetivos específicos",
         bool(config["pregunta"]) and 3 <= len(config["objetivos_especificos"]) <= 5),
        ("Alcance, restricciones y supuestos declarados por separado",
         all(config[k] for k in ("alcance", "restricciones", "supuestos"))),
        ("Kernel del proyecto (.venv)", ".venv" in sys.executable),
        ("Carpetas del proyecto presentes", all((raiz / c).is_dir() for c in CARPETAS)),
        ("requirements.txt con versiones fijas", len(leer_requirements(raiz / "requirements.txt")) > 0),
        ("Módulo src/utilidades.py importable", (raiz / "src" / "utilidades.py").is_file()),
        ("Requisitos obligatorios del dataset cumplidos", evaluacion.query("tipo == 'obligatorio'")["cumple"].all()),
        ("Todas las variables con rol analítico", (diccionario["rol_analitico"] != "sin clasificar").all()),
    ]
    for texto, ok in comprobaciones:
        print(("[OK]   " if ok else "[FALLA]"), texto)
        assert ok, texto
    print(f"\nFase 1 validada: {len(comprobaciones)} de {len(comprobaciones)} comprobaciones")

validar_fase1(RAIZ, PROYECTO, evaluacion, diccionario)

[OK]    Problema con pregunta, objetivo general y 3 a 5 objetivos específicos
[OK]    Alcance, restricciones y supuestos declarados por separado
[OK]    Kernel del proyecto (.venv)
[OK]    Carpetas del proyecto presentes
[OK]    requirements.txt con versiones fijas
[OK]    Módulo src/utilidades.py importable
[OK]    Requisitos obligatorios del dataset cumplidos
[OK]    Todas las variables con rol analítico

Fase 1 validada: 8 de 8 comprobaciones


## 8. Vinculación con el mapa conceptual

La rúbrica (criterio 14) pide relacionar cada elemento del mapa conceptual con lo implementado, en
los **seis tipos** de elemento. Hay que decir qué quedó implementado y qué queda proyectado para las
fases siguientes. Cada fila apunta a una evidencia que se puede verificar en el repositorio.

In [20]:
VINCULACION = [
    # (elemento del mapa, tipo de elemento, estado, evidencia en el repositorio) · estado al cierre de la Fase 3
    ("Python 3.14 + .venv + requirements.txt", "herramientas", "implementado", "Sección 2 · requirements.txt con 6 librerías, instalación probada en Linux, macOS y Windows"),
    ("JupyterLab y kernel «Python (mcdi500)»", "herramientas", "implementado", "Sección 2 · celda de configuración de F2 y F3"),
    ("pandas, NumPy, scikit-learn, matplotlib", "herramientas", "implementado", "F2 y F3 · docs/figuras"),
    ("Git y GitHub, una sola identidad", "herramientas", "implementado", "Historial de commits · docs/bitacora_proceso.md"),
    ("data/raw → F1 → F2 → F3 → data/processed", "flujo reproducible", "implementado", "Huella SHA-256 y metadatos de F1, F2 y F3"),
    ("F3 · algoritmos, eficiencia y POO", "flujo reproducible", "implementado", "src/ y tests/ · F3/F3_Nucleo_Eficiencia_POO.ipynb"),
    ("F4 · visualización y comunicación", "flujo reproducible", "proyectado", "Sumativa 3"),
    ("Anonimizar personas y sitios", "procesos", "implementado", "scripts/anonimizar_incidencias.py · bitácora pasos 4 y 25"),
    ("Medir calidad sin corregir", "procesos", "implementado", "Secciones 5.3 a 5.5 de este notebook"),
    ("Limpiar, transformar y escalar", "procesos", "implementado", "F2, secciones 3 a 5 · clases de src/transformadores.py (F3)"),
    ("Recurrencia por sitio con fusión de intervalos", "procesos", "implementado", "F2, sección 4.4 · F3, sección 6.1 (versión iterativa)"),
    ("Uptime = P1, P2, P3 y P3E, sin canceladas, solo octubre", "decisiones técnicas", "implementado", "Sección 1 (supuesto 5) · F2, secciones 4.1 y 4.4"),
    ("Duración vacía no se inventa en el limpio", "decisiones técnicas", "implementado", "F2, sección 3.6 · F3, sección 9 (limitación declarada)"),
    ("Un sitio por fila (dos espacios = Q sitios)", "decisiones técnicas", "implementado", "Sección 5.5 · regla de integridad en F2"),
    ("StandardScaler y MinMaxScaler según la variable", "decisiones técnicas", "implementado", "F2, sección 5 · Escalador con patrón Strategy (F3)"),
    ("Datos del operador: sitios con código y repo privado", "decisiones técnicas", "implementado", "Ficha 5 · bitácora pasos 24 y 25"),
    ("F1/ a F4/, data/, docs/, src/, scripts/, tests/, README", "organización del repositorio", "implementado", "Sección 3 · README generado por este notebook"),
    ("src/ en módulos con una responsabilidad cada uno", "organización del repositorio", "implementado", "carga, limpieza, transformadores, pipeline, medicion y utilidades"),
    ("Ramas por componente y pull request", "organización del repositorio", "implementado", "f2-pipeline y los pull request #1 a #3 de la Fase 3"),
    ("Celda narrativa → código → evidencia", "estructura del notebook", "implementado", "F1, F2 y F3 completos"),
    ("Pruebas normal, límite y excepción", "estructura del notebook", "implementado", "Sección 4.1 · F2 sección 6.2 · tests/ (30 pruebas)"),
]
vinculacion = pd.DataFrame(VINCULACION, columns=["elemento_mapa", "tipo", "estado", "evidencia"])
tipos_exigidos = {"herramientas", "procesos", "flujo reproducible", "decisiones técnicas",
                  "organización del repositorio", "estructura del notebook"}
assert tipos_exigidos <= set(vinculacion["tipo"]), "Falta algún tipo de elemento del mapa"
print("Elementos por estado:", vinculacion["estado"].value_counts().to_dict())
vinculacion

Elementos por estado: {'implementado': 20, 'proyectado': 1}


,elemento_mapa,tipo,estado,evidencia
0,Python 3.14 + .venv + requirements.txt,herramientas,implementado,"Sección 2 · requirements.txt con 6 librerías, ..."
1,JupyterLab y kernel «Python (mcdi500)»,herramientas,implementado,Sección 2 · celda de configuración de F2 y F3
2,"pandas, NumPy, scikit-learn, matplotlib",herramientas,implementado,F2 y F3 · docs/figuras
3,"Git y GitHub, una sola identidad",herramientas,implementado,Historial de commits · docs/bitacora_proceso.md
4,data/raw → F1 → F2 → F3 → data/processed,flujo reproducible,implementado,"Huella SHA-256 y metadatos de F1, F2 y F3"
5,"F3 · algoritmos, eficiencia y POO",flujo reproducible,implementado,src/ y tests/ · F3/F3_Nucleo_Eficiencia_POO.ipynb
6,F4 · visualización y comunicación,flujo reproducible,proyectado,Sumativa 3
7,Anonimizar personas y sitios,procesos,implementado,scripts/anonimizar_incidencias.py · bitácora p...
8,Medir calidad sin corregir,procesos,implementado,Secciones 5.3 a 5.5 de este notebook
9,"Limpiar, transformar y escalar",procesos,implementado,"F2, secciones 3 a 5 · clases de src/transforma..."


## 9. Persistencia y trazabilidad

**Qué hace este paso.** Guarda en `docs/` los productos de la fase, para que el informe los cite y
cualquiera los verifique. También genera el README desde el diccionario `PROYECTO`, así el README no
puede quedar desactualizado respecto del notebook (sugerencia del docente, lámina 68).

In [21]:
diccionario.to_csv(DIR_DOCS / "diccionario_variables.csv", index=False, encoding="utf-8")
evaluacion.to_csv(DIR_DOCS / "evaluacion_criterios_dataset.csv", index=False, encoding="utf-8")
vinculacion.to_csv(DIR_DOCS / "vinculacion_mapa_conceptual.csv", index=False, encoding="utf-8")
problemas.to_csv(DIR_DOCS / "problemas_calidad_f1.csv", index=False, encoding="utf-8")

_, commit_actual = git("rev-parse", "--short", "HEAD")
metadatos = {
    "proyecto": PROYECTO["titulo"], "fase": "F1", "fecha_ejecucion": date.today().isoformat(),
    "commit_base": commit_actual, "semilla": SEMILLA, "python": platform.python_version(),
    "sistema": f"{platform.system()} {platform.release()}", "entorno_virtual": ".venv" in sys.executable,
    "dependencias": dict(zip(entorno["paquete"], entorno["instalada"])),
    "dataset": {"archivo": FICHA["archivo"], "filas": int(df_crudo.shape[0]), "columnas": int(df_crudo.shape[1]),
                "duraciones_vacias": int(duracion_vacia.sum()), "fallas_transmision": int(len(tx)),
                "elementos_tx_con_2_o_mas": int((fallas_por_elemento >= 2).sum()),
                "fallas_tx_con_perdida_de_servicio": int(afecta_uptime.sum()),
                "elementos_con_2_o_mas_perdidas": int((fallas_uptime_por_elemento >= 2).sum())},
}
(DIR_DOCS / "metadatos_fase1.json").write_text(json.dumps(metadatos, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(metadatos, ensure_ascii=False, indent=2))

{
  "proyecto": "Fallas recurrentes de transmisión en la red de un operador móvil",
  "fase": "F1",
  "fecha_ejecucion": "2026-11-17",
  "commit_base": "c25abe5",
  "semilla": 42,
  "python": "3.14.6",
  "sistema": "Windows 11",
  "entorno_virtual": true,
  "dependencias": {
    "numpy": "2.5.4",
    "pandas": "3.1.1",
    "matplotlib": "3.11.2",
    "scikit-learn": "1.9.1",
    "jupyterlab": "4.6.6",
    "ipykernel": "7.4.0"
  },
  "dataset": {
    "archivo": "data/raw/incidencias_oct2026_anon.csv",
    "filas": 21542,
    "columnas": 49,
    "duraciones_vacias": 5318,
    "fallas_transmision": 588,
    "elementos_tx_con_2_o_mas": 366,
    "fallas_tx_con_perdida_de_servicio": 264,
    "elementos_con_2_o_mas_perdidas": 151
  }
}


In [22]:
def generar_readme(config, ficha, metadatos, ruta):
    '''Escribe el README técnico a partir de la definición del proyecto.'''
    objetivos = "\n".join(f"{i}. {o}" for i, o in enumerate(config["objetivos_especificos"], 1))
    filas_txt = f"{metadatos['dataset']['filas']:,}".replace(",", ".")   # 21.542: separador de miles chileno
    texto = f'''# {config["titulo"]}

Proyecto del curso **{config["asignatura"]}** · Magíster en Ciencia de Datos e Inteligencia Artificial, Universidad Andrés Bello.

- **Integrante:** {", ".join(config["integrantes"])} ([@{config["github"]}](https://github.com/{config["github"]})) · {config["grupo"]}
- **Docente:** {config["docente"]}

## Problemática

{config["situacion"]}

**Pregunta:** {config["pregunta"]}

**Objetivo general:** {config["objetivo_general"]}

**Objetivos específicos:**

{objetivos}

**Metodología:** {config["metodologia"]}

## Datos

- **Origen:** {ficha["origen"]}. {ficha["permiso"]}.
- **Período:** {ficha["periodo"]}.
- **Archivo:** `{ficha["archivo"]}` ({filas_txt} filas × {metadatos["dataset"]["columnas"]} columnas, separador `{ficha["separador"]}`).
- **Anonimización:** {ficha["anonimizacion"]}. El exporte original no forma parte del repositorio.

## Estructura

```
data/raw/        datos de entrada anonimizados (no se modifican)
data/processed/  resultado de la limpieza y transformación (F2; F3 lo reproduce idéntico)
docs/            diccionario de variables, bitácoras, metadatos, mediciones y vinculación con el mapa
src/             código del proyecto (ver «Arquitectura del código»)
tests/           pruebas de casos normales, límite y excepciones (unittest)
scripts/         preparación previa de los datos (anonimización)
F1/ F2/ F3/ F4/  un notebook por fase
```

## Arquitectura del código (Fase 3)

| Módulo | Responsabilidad | Usa a |
|---|---|---|
| `src/utilidades.py` | Funciones base de las Fases 1 y 2: texto, duraciones, sitios y fusión de intervalos (recursiva e iterativa) | — |
| `src/carga.py` | Lee `data/raw` y verifica que sea el archivo que registró la Fase 1 | utilidades |
| `src/limpieza.py` | Operaciones que **no** aprenden de los datos (funciones) | utilidades |
| `src/transformadores.py` | Pasos que **sí** aprenden de los datos: clase base `Transformador` y cinco clases hijas, más el adaptador `PasoFijo` | utilidades |
| `src/pipeline.py` | `Pipeline` que encadena los 15 pasos, `Bitacora` (patrón Observer) y `ejecutar`, que repite la Fase 2 con una llamada | carga, limpieza, transformadores |
| `src/medicion.py` | Compara implementaciones en tiempo (`timeit`) y memoria (`tracemalloc`) | — |

Patrones de diseño: **Observer** (la bitácora registra cada paso sin que el paso lo sepa) y **Strategy** (el `Escalador` recibe la forma de escalar como parámetro).

```python
from src.pipeline import construir_pipeline, ejecutar
resultado = ejecutar(RAIZ)                                   # la Fase 2 completa
tubo = construir_pipeline().ajustar(entrenamiento)          # Fase 4: ajustar solo con entrenamiento
prueba_lista = tubo.transformar(prueba)
```

## Cómo ejecutar el proyecto desde cero (Windows, PowerShell)

```
git clone {config["repositorio"]}.git
cd mcdi500-grupo6
python -m venv .venv
.\\.venv\\Scripts\\python.exe -m pip install -r requirements.txt
.\\.venv\\Scripts\\python.exe -m ipykernel install --user --name mcdi500 --display-name "Python (mcdi500)"
.\\.venv\\Scripts\\python.exe -m jupyter lab
```

En JupyterLab, abrir los notebooks en orden (`F1/F1_Definicion.ipynb`, `F2/F2_Preprocesamiento.ipynb` y `F3/F3_Nucleo_Eficiencia_POO.ipynb`), elegir el kernel **Python (mcdi500)** y ejecutar *Kernel → Restart Kernel and Run All Cells*. En macOS o Linux se usa `.venv/bin/python` en lugar de `.\\.venv\\Scripts\\python.exe`.

Las pruebas se ejecutan con un solo comando desde la raíz del proyecto (unittest viene con Python):

```
.\\.venv\\Scripts\\python.exe -m unittest discover -s tests -v
```

## Reproducibilidad

- Python {metadatos["python"]} en entorno virtual `.venv`.
- `requirements.txt` declara solo las librerías del proyecto, con versión fija; pip resuelve el resto según el sistema operativo.
- La instalación se prueba en Linux, macOS y Windows con GitHub Actions (`.github/workflows/instalacion.yml`) en cada push.
- Semilla aleatoria: {metadatos["semilla"]}. Rutas relativas a la raíz del proyecto (`localizar_raiz`).
- F1 verifica el entorno en su sección 2: se detiene si el kernel no es el del proyecto e informa las versiones que no coincidan.
- F3 comprueba que el pipeline con clases produce los mismos archivos que F2, byte a byte (huella SHA-256); la prueba `test_normal_mismos_archivos_que_la_fase_2` lo repite en cada ejecución de las pruebas.

## Convención de commits

`docs:` documentación · `data:` datos · `feat:` código nuevo · `fix:` corrección · `test:` pruebas · `perf:` cambio por rendimiento medido. El historial con el detalle de cada commit y el criterio de la rúbrica que respalda está en `docs/bitacora_proceso.md`.

## Uso de herramientas de inteligencia artificial

Se utilizó un asistente de inteligencia artificial generativa como apoyo para explicar conceptos y revisar código. Las decisiones, la ejecución de los notebooks, los commits y la revisión final son del autor.
'''
    ruta.write_text(texto, encoding="utf-8")
    return ruta

print("README generado:", generar_readme(PROYECTO, FICHA, metadatos, RAIZ / "README.md").relative_to(RAIZ))

README generado: README.md


## 10. Conclusiones y reflexión técnica

- **El conjunto cumple los requisitos del curso** (sección 5.1) y trae problemas reales que dan
  material a la Fase 2: miles de celdas con tildes rotas, la misma categoría escrita de varias
  formas, columnas casi vacías y tickets repetidos (sección 5.3).
- **El hallazgo más importante es el de las duraciones vacías** (sección 5.4). No son datos perdidos
  al azar: son incidencias que aún no se normalizan. Eso condiciona la Fase 2, porque rellenarlas
  con un valor central inventaría cierres que no ocurrieron.
- **La pregunta tiene datos para responderse** (sección 5.5): hay 588 fallas de transmisión; 264
  dejaron el servicio caído (P1, P2, P3 o P3E). Contando cada sitio por separado, 561 sitios
  perdieron servicio y 151 lo perdieron 2 o más veces en el mes; el que más, SIT_0535, 11 veces.
  Esa recurrencia es la que justifica escalar con las áreas de transmisión y evaluar cambios de
  enlace.
- **Una incidencia puede tumbar muchos sitios.** 1.098 incidencias traen varios sitios en la misma
  celda. Contarlas como un solo "elemento" escondía cuáles sitios caen de verdad.
- **Contar solo la pérdida de servicio cambia la lista.** El sitio con más fallas de transmisión,
  SIT_0304, tiene 19, pero solo 1 dejó el servicio caído; las otras fueron pérdidas de
  redundancia o P4. Sin ese filtro se escalarían sitios equivocados.
- **Hay 2 duraciones negativas** en incidencias cerradas (sección 5.3). La Fase 2 decide cómo
  tratarlas antes de calcular minutos acumulados.
- **El entorno quedó verificado con evidencia**: el kernel del proyecto y las versiones que coinciden
  con `requirements.txt`. El módulo propio quedó probado con casos normales, límite y excepciones.

**Queda proyectado:** la limpieza, la imputación, la transformación y el escalamiento (Fase 2); el
núcleo algorítmico (Fase 3); la visualización y la comunicación (Fase 4).

## 11. Verificación antes de entregar

- [x] Corre completo tras *Kernel → Restart Kernel and Run All Cells*, sin errores.
- [x] La numeración de ejecución es continua (1, 2, 3, ...).
- [x] Cada bloque de código está precedido por una celda de texto que explica qué hace y por qué.
- [x] No limpia, no imputa y no transforma: solo define, verifica y mide.
- [x] Las salidas quedaron guardadas: no se borran antes de entregar.
- [x] Los archivos de `docs/` y el README se versionan en el mismo commit que este notebook.